# Módulo 10 · Aula 02 — Pandas Essencial

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O relatório em Pandas dá um número. A consulta SQL dá outro. A planilha do financeiro dá um terceiro. Ninguém sabe qual está certo — e a diferença é de R$ 40 mil."*

Spoiler: os três "estão certos". Eles respondem perguntas **ligeiramente diferentes** sem que ninguém tenha percebido.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Series e DataFrame | O modelo mental |
| 2 | `loc` vs `iloc` | E a confusão que causa bug |
| 3 | Filtros | Máscaras booleanas |
| 4 | 🔴 **`SettingWithCopy`** | O aviso que todo mundo ignora |
| 5 | Nulos | 🔴 A causa nº 1 de número errado |
| 6 | Duplicatas | E o que "duplicata" significa |
| 7 | **Vetorização** | 🎯 `apply` é um `for` disfarçado |
| 8 | `groupby` | O padrão dividir-aplicar-combinar |
| 9 | `merge` | E os quatro jeitos de perder linhas |
| 10 | Séries temporais | `resample` e janelas móveis |

> 🎯 **Esta aula é sobre as armadilhas.** A sintaxe do Pandas você acha na documentação. O que quase não se documenta é **por que o seu número saiu diferente do SQL** — e é isso que faz relatório perder credibilidade.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

## 1. Series e DataFrame

In [ ]:
BASE = preparar("aula_10_02")
vendas = gerar_vendas(n=50_000, dias=180)

print(f"DataFrame: {vendas.shape[0]:,} linhas × {vendas.shape[1]} colunas\n")
print(vendas.head(3).to_string(index=False))

print("\n💭 O MODELO MENTAL\n")
print("   Series    uma coluna + um ÍNDICE (não é uma lista)")
print("   DataFrame um dicionário de Series que compartilham o índice")
print("\n   🔑 O ÍNDICE é o que diferencia o Pandas de uma lista de dicionários.")
print("      Ele alinha as operações — e é a origem de metade dos bugs.")

In [ ]:
# 🔑 O alinhamento por índice em ação
a = pd.Series([10, 20, 30], index=["x", "y", "z"])
b = pd.Series([1, 2, 3], index=["z", "y", "w"])

print("a:", dict(a))
print("b:", dict(b))
print("\na + b:")
print((a + b).to_string())

print("""
🔴 REPARE: o Pandas somou por RÓTULO, não por posição.

   · 'y' existe nos dois → 20 + 2 = 22
   · 'z' existe nos dois → 30 + 1 = 31
   · 'x' e 'w' só existem em um → NaN

   Se você esperava [11, 22, 33], errou o modelo mental — e é assim
   que uma soma vira NaN sem nenhum erro aparecer.
""")

In [ ]:
# dtypes: o que o Pandas achou que os seus dados são
print(vendas.dtypes.to_string())
print(f"\nmemória: {tamanho(memoria(vendas))}")
print("""
⚠️ `object` significa "não sei" — na prática, ponteiros para objetos
   Python espalhados na memória. É o dtype mais lento e mais pesado.

   As colunas de texto acima são `object`. Na aula 10_03 você vai ver
   quanto isso custa — e como resolver.
""")

## 2. `loc` vs `iloc`

In [ ]:
amostra = vendas.head(5).copy()
amostra.index = [10, 20, 30, 40, 50]      # índice que NÃO é 0,1,2...

print("Um DataFrame com índice 10, 20, 30, 40, 50:\n")
print(amostra[["pedido_id", "sku", "quantidade"]].to_string())

print("\n" + "─" * 60)
print("loc[20]  → pelo RÓTULO do índice")
print(f"   {amostra.loc[20, 'sku']}")
print("\niloc[20] → pela POSIÇÃO (falha: só há 5 linhas)")
try:
    amostra.iloc[20]
except IndexError as erro:
    print(f"   🔴 IndexError: {erro}")
print("\niloc[1]  → a SEGUNDA linha")
print(f"   {amostra.iloc[1]['sku']}")

In [ ]:
print("""
   loc[rotulo]        pelo rótulo do índice
   iloc[posicao]      pela posição (0, 1, 2...)

   loc[linhas, colunas]     ambos por rótulo
   iloc[linhas, colunas]    ambos por posição
""")

print("⚠️ E a diferença que pega todo mundo — as FATIAS:\n")
print(f"   loc[10:30]  → {len(amostra.loc[10:30])} linhas  (INCLUI o 30)")
print(f"   iloc[0:3]   → {len(amostra.iloc[0:3])} linhas  (exclui o 3)")
print("""
   🔑 `loc` inclui o fim porque trabalha com RÓTULOS — e um rótulo é
      um nome, não uma posição. `iloc` segue a convenção do Python.

   💭 Não é inconsistência: são duas semânticas diferentes. Mas é
      preciso saber qual você está usando.
""")

## 3. Filtros

In [ ]:
# Máscara booleana: o pão com manteiga do Pandas
mascara = vendas["status"] == "pago"
print(f"máscara: Series de booleanos, {mascara.sum():,} verdadeiros "
      f"de {len(mascara):,}\n")

pagos = vendas[mascara]
print(f"vendas[mascara] → {len(pagos):,} linhas")

# Várias condições
caros_pagos = vendas[
    (vendas["status"] == "pago")
    & (vendas["preco_unitario"] > 2000)
    & (vendas["cidade"].isin(["Campinas", "São Paulo"]))
]
print(f"três condições  → {len(caros_pagos):,} linhas")

print("""
🔴 AS DUAS ARMADILHAS DA MÁSCARA

   1. Use `&` `|` `~`, NÃO `and` `or` `not`.
      `and` opera em UM booleano; a máscara é um vetor.

   2. PARÊNTESES em toda condição.
      `&` tem precedência MAIOR que `==` em Python — sem parênteses,
      a expressão é agrupada errado.
""")

try:
    vendas[vendas["status"] == "pago" & vendas["quantidade"] > 2]
except TypeError as erro:
    print(f"   sem parênteses → 🔴 TypeError: {str(erro)[:60]}...")

In [ ]:
# `query`: às vezes mais legível
resultado = vendas.query("status == 'pago' and preco_unitario > 2000")
print(f"query() → {len(resultado):,} linhas (mesmo resultado)")
print("""
💡 `query()` aceita `and`/`or` porque é uma expressão em texto —
   ela não sofre o problema de precedência.

⚠️ Mas ela é mais lenta e não tem verificação do editor. Use para
   filtros longos e legíveis; máscara para o resto.
""")

## 4. 🔴 `SettingWithCopyWarning`

In [ ]:
print("""
🔴 A ALTERAÇÃO QUE NÃO ACONTECE — E NÃO AVISA.
""")

# O caminho errado
df = vendas.head(1000).copy()
recorte = df[df["cidade"] == "Campinas"]      # 🔴 cópia ou vista? não se sabe

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    recorte["desconto"] = 0.1
    houve_aviso = any("SettingWithCopy" in str(a.message) for a in avisos)

print(f"   a coluna entrou no RECORTE?   {'desconto' in recorte.columns}")
print(f"   a coluna entrou no ORIGINAL?  {'desconto' in df.columns}")
print(f"   o pandas avisou?              {houve_aviso}")

print(f"""
🔴 A INTENÇÃO ERA MARCAR AS VENDAS DE CAMPINAS NO `df`. NÃO ACONTECEU.

   O `df` original não tem a coluna. A alteração foi para um objeto
   intermediário que ninguém guardou — e o programa seguiu adiante
   sem erro nenhum.

⚠️ E repare que o aviso NÃO apareceu (`{houve_aviso}`).

   O `SettingWithCopyWarning` é famoso, mas nas versões recentes do
   pandas ele nem sempre é emitido. Contar com ele é contar com sorte.

🔑 A CAUSA É A AMBIGUIDADE.

   `df[mascara]` pode devolver uma CÓPIA ou uma VISTA do original —
   o pandas não garante qual. Se for cópia, a sua alteração some
   (foi o que aconteceu). Se for vista, ela altera o original sem
   você pedir.

   Você não deveria precisar adivinhar — e a regra abaixo elimina a
   dúvida.
""")

In [ ]:
print("✅ AS DUAS FORMAS CORRETAS\n")

# 1. Se você quer alterar o ORIGINAL → use .loc
df1 = vendas.head(1000).copy()
df1.loc[df1["cidade"] == "Campinas", "desconto"] = 0.1
print(f"1. .loc[mascara, coluna] = valor")
print(f"   linhas com desconto no original: {df1['desconto'].notna().sum()}")

# 2. Se você quer um NOVO DataFrame → use .copy()
df2 = vendas.head(1000).copy()
recorte = df2[df2["cidade"] == "Campinas"].copy()      # 🔑 .copy() explícito
recorte["desconto"] = 0.1
print(f"\n2. df[mascara].copy()  →  altere o recorte à vontade")
print(f"   no recorte: {'desconto' in recorte.columns}   "
      f"no original: {'desconto' in df2.columns}")

print("""
🧭 A REGRA:

   quer alterar o original?  →  .loc[mascara, coluna] = valor
   quer um novo objeto?      →  df[mascara].copy()

   Nunca `df[mascara]["coluna"] = ...` — isso é INDEXAÇÃO ENCADEADA,
   e é exatamente o caso ambíguo.

💡 O Pandas 3.0 adota "copy-on-write" e resolve a ambiguidade de vez.
   Até lá — e para ler código antigo — a regra acima vale.
""")

## 5. 🔴 Nulos — a causa nº 1 de número errado

In [ ]:
# Sujamos os dados como a realidade suja
sujo = vendas.head(10_000).copy()
rng_local = np.random.default_rng(SEMENTE)
sujo.loc[rng_local.choice(sujo.index, 400, replace=False), "frete"] = np.nan
sujo.loc[rng_local.choice(sujo.index, 250, replace=False), "cidade"] = None
sujo.loc[rng_local.choice(sujo.index, 120, replace=False), "quantidade"] = np.nan

print("Nulos por coluna:\n")
nulos = sujo.isna().sum()
for coluna, n in nulos[nulos > 0].items():
    print(f"   {coluna:<16}{n:>6,}  ({n / len(sujo) * 100:.1f}%)")

In [ ]:
print("🔴 E AGORA A PARTE QUE FAZ O NÚMERO SAIR DIFERENTE DO SQL:\n")

frete = sujo["frete"]
print(f"   len(frete)          {len(frete):>10,}")
print(f"   frete.count()       {frete.count():>10,}   ← ignora NaN")
print(f"   frete.sum()         {frete.sum():>10,.2f}   ← ignora NaN")
print(f"   frete.mean()        {frete.mean():>10,.2f}   ← soma/COUNT, não /len")
print(f"   soma/len            {frete.sum() / len(frete):>10,.2f}   ← ⚠️ outro número")

print("""
🎯 AQUI ESTÁ A DIFERENÇA DE R$ 40 MIL DA AURORA.

   `mean()` divide pela contagem de NÃO-NULOS. Quem calcula
   "soma dividida pelo total de linhas" obtém outro valor — e os dois
   se chamam "média".

   ⚠️ O SQL faz o mesmo: `AVG(coluna)` ignora NULL. Mas a planilha do
      financeiro, com células vazias tratadas como zero, NÃO.

   🧭 A regra: antes de comparar dois números, pergunte
      "o que cada um fez com os nulos?"
""")

In [ ]:
# As operações que se comportam diferente
print("Como cada operação trata o nulo:\n")
comportamentos = [
    ["sum(), mean(), max()", "IGNORA",       "skipna=True é o padrão"],
    ["count()",              "IGNORA",       "conta só os preenchidos"],
    ["len(), shape",         "CONTA",        "são linhas, não valores"],
    ["groupby(chave)",       "🔴 DESCARTA",  "linhas com chave nula somem"],
    ["==  entre nulos",      "🔴 False",     "NaN != NaN"],
    ["sort_values()",        "vão para o fim", "na_position='last'"],
    ["merge(on=chave)",      "🔴 não casa",  "nulo não casa com nulo"],
]
tabela(["OPERAÇÃO", "COM NULO", "OBSERVAÇÃO"], comportamentos, [24, 16, 32])

print("\n🔴 A TERCEIRA LINHA É A MAIS PERIGOSA:\n")
por_cidade = sujo.groupby("cidade", observed=True)["frete"].sum()
print(f"   linhas no total         : {len(sujo):>7,}")
print(f"   soma dos grupos         : {por_cidade.sum():>10,.2f}")
print(f"   soma da coluna inteira  : {sujo['frete'].sum():>10,.2f}")
print(f"   🔴 diferença            : "
      f"{sujo['frete'].sum() - por_cidade.sum():>10,.2f}")
print("\n   As 250 linhas com cidade nula sumiram do agrupamento —")
print("   e o total do relatório não bate com o total da tabela.")

In [ ]:
print("✅ E a correção é uma opção:\n")
com_nulos = sujo.groupby("cidade", observed=True, dropna=False)["frete"].sum()
print(f"   dropna=False → {len(com_nulos)} grupos "
      f"(inclui o grupo NaN)")
print(f"   soma dos grupos: {com_nulos.sum():,.2f}  ✅ agora bate")

print("""
🧭 Tratamento de nulo é DECISÃO DE NEGÓCIO, não técnica:

   frete nulo       → é zero (retirada na loja)? ou é desconhecido?
   cidade nula      → agrupa em "não informado"? ou descarta?
   quantidade nula  → 🔴 nunca invente. Isso é dado corrompido.

   💡 Escreva a decisão no código, com um comentário dizendo por quê.
      Daqui a seis meses ninguém vai lembrar.
""")

In [ ]:
# As ferramentas
limpo = sujo.copy()
limpo["frete"] = limpo["frete"].fillna(0)                    # decisão: é zero
limpo["cidade"] = limpo["cidade"].fillna("não informado")    # decisão: agrupa
antes = len(limpo)
limpo = limpo.dropna(subset=["quantidade"])                  # decisão: descarta

print(f"   fillna(0) no frete           → {limpo['frete'].isna().sum()} nulos")
print(f"   fillna('não informado')      → {limpo['cidade'].isna().sum()} nulos")
print(f"   dropna(subset=['quantidade'])→ {antes:,} - {antes - len(limpo)} = {len(limpo):,}")

print("\n🔴 E as linhas descartadas NÃO SOMEM: elas vão para a quarentena.")
print("   (o padrão que você vai construir na aula 10_05)")

## 6. Duplicatas

In [ ]:
com_duplicatas = pd.concat([vendas.head(5_000), vendas.head(300)], ignore_index=True)

print(f"linhas: {len(com_duplicatas):,}\n")
print(f"   duplicated()                    → "
      f"{com_duplicatas.duplicated().sum():>5,} linhas idênticas")
print(f"   duplicated(subset=['pedido_id'])→ "
      f"{com_duplicatas.duplicated(subset=['pedido_id']).sum():>5,} ids repetidos")

print("""
🔑 "DUPLICATA" DEPENDE DA CHAVE DE NEGÓCIO.

   Duas linhas idênticas podem ser:
   · o mesmo pedido importado duas vezes        → 🔴 remova
   · o mesmo cliente comprando o mesmo item     → ✅ legítimo

   Sem uma CHAVE NATURAL definida, você não sabe qual é qual.
   Para a Aurora, `pedido_id` é a chave — dois registros com o mesmo
   id são o mesmo evento.
""")

sem_duplicatas = com_duplicatas.drop_duplicates(subset=["pedido_id"], keep="last")
print(f"drop_duplicates(subset=['pedido_id'], keep='last')")
print(f"   {len(com_duplicatas):,} → {len(sem_duplicatas):,} linhas")

print("""
⚠️ `keep='last'` importa: se o registro foi CORRIGIDO e reenviado, a
   última versão é a boa. `keep='first'` manteria a errada.

   💭 E isso exige que a ordem esteja garantida — ordene por data de
      ingestão antes, senão "last" é o que o acaso decidir.
""")

## 7. 🎯 Vetorização — `apply` é um `for` disfarçado

In [ ]:
teste = vendas.head(50_000).copy()

def com_loop():
    saida = []
    for _, linha in teste.iterrows():
        saida.append(linha["quantidade"] * linha["preco_unitario"])
    return saida

def com_apply():
    return teste.apply(lambda l: l["quantidade"] * l["preco_unitario"], axis=1)

def com_vetorizado():
    return teste["quantidade"] * teste["preco_unitario"]

def com_numpy():
    return teste["quantidade"].to_numpy() * teste["preco_unitario"].to_numpy()

print("A MESMA conta, quatro jeitos (50 mil linhas):\n")
comparar([("for + iterrows()", com_loop),
          ("apply(axis=1)", com_apply),
          ("vetorizado (Series)", com_vetorizado),
          ("NumPy puro", com_numpy)], rotulo="abordagem")

> 🎯 **A diferença é de ordens de grandeza, e a razão é simples.**
>
> `iterrows()` e `apply(axis=1)` executam **Python interpretado uma vez por linha** — e ainda constroem um objeto `Series` para cada uma.
>
> A versão vetorizada chama **uma** rotina em C que percorre o array inteiro. É a mesma diferença entre pedir 50 mil vezes "some estes dois números" e pedir uma vez "some estes dois arrays".
>
> 🧭 **A regra:** se você escreveu `for` ou `apply(axis=1)` num DataFrame, pare e procure a versão vetorizada. Ela quase sempre existe.
>
> ⚠️ **Quando `apply` se justifica:** lógica que realmente não vetoriza (chamar uma API por linha, aplicar uma regra complexa com estado). E aí a pergunta seguinte é se aquilo deveria mesmo estar num DataFrame.

In [ ]:
# Condicionais vetorizadas
print("Condicional sem `apply`:\n")

teste["faixa"] = np.where(teste["preco_unitario"] > 2000, "alto", "baixo")
print("   np.where(condicao, se_sim, se_nao)      → 2 opções")

teste["curva"] = np.select(
    [teste["preco_unitario"] > 2500,
     teste["preco_unitario"] > 800],
    ["A", "B"], default="C")
print("   np.select([cond1, cond2], [v1, v2], ...) → N opções")

teste["categoria_curta"] = teste["categoria"].map(
    {"Notebooks": "NB", "Monitores": "MO",
     "Periféricos": "PE", "Armazenamento": "AR"})
print("   Series.map(dicionario)                   → tradução")

print("\n" + teste[["preco_unitario", "faixa", "curva", "categoria_curta"]]
      .head(5).to_string(index=False))

## 8. `groupby` — dividir, aplicar, combinar

In [ ]:
pagas = vendas[vendas["status"] == "pago"].copy()
pagas["receita"] = pagas["quantidade"] * pagas["preco_unitario"]
pagas["margem"] = pagas["quantidade"] * (
    pagas["preco_unitario"] - pagas["custo_unitario"])

resumo = (pagas.groupby("categoria", observed=True)
          .agg(pedidos=("pedido_id", "nunique"),
               itens=("quantidade", "sum"),
               receita=("receita", "sum"),
               margem=("margem", "sum"),
               ticket=("receita", "mean"))
          .sort_values("receita", ascending=False))

print(resumo.to_string())

print("""
💡 A FORMA NOMEADA É A MELHOR:

     .agg(nome_da_saida=("coluna", "funcao"))

   Ela deixa explícito de onde vem cada número e já nomeia a coluna —
   sem MultiIndex para desembrulhar depois.
""")

In [ ]:
# Agrupamento por várias chaves
por_mes = (pagas.assign(mes=pagas["data"].dt.to_period("M").astype(str))
           .groupby(["mes", "canal"], observed=True)
           .agg(receita=("receita", "sum"))
           .reset_index())

print("Receita por mês e canal:\n")
print(por_mes.pivot(index="mes", columns="canal", values="receita").to_string())

print("""
💡 `pivot` transforma linhas em colunas — é o formato que humano lê.
   O formato LONGO (uma linha por combinação) é o que máquina prefere.

   🔑 `reset_index()` depois do groupby: sem ele, as chaves viram
      índice e a próxima operação te surpreende.
""")

In [ ]:
# 🔑 `transform`: agregado SEM perder as linhas
pagas["receita_da_categoria"] = pagas.groupby("categoria", observed=True)["receita"].transform("sum")
pagas["participacao"] = (pagas["receita"] / pagas["receita_da_categoria"] * 100).round(3)

print("`transform` devolve uma Series do MESMO tamanho:\n")
print(pagas[["sku", "categoria", "receita", "receita_da_categoria", "participacao"]]
      .head(5).to_string(index=False))

print("""
🔑 `agg` REDUZ (n linhas → 1 por grupo).
   `transform` PRESERVA (n linhas → n linhas, com o valor do grupo).

   💡 `transform` é o que você quer para "qual a participação desta
      venda no total da categoria?" — sem precisar de merge.
""")

## 9. `merge` — e os quatro jeitos de perder linhas

In [ ]:
produtos = gerar_produtos()
# Simulamos o mundo real: o cadastro não tem TODOS os SKUs vendidos
produtos_incompleto = produtos.head(50)

esquerda = vendas.head(5_000)
print(f"vendas   : {len(esquerda):,} linhas, "
      f"{esquerda['sku'].nunique()} SKUs distintos")
print(f"produtos : {len(produtos_incompleto)} linhas\n")

for tipo in ["inner", "left", "right", "outer"]:
    resultado = esquerda.merge(produtos_incompleto, on="sku", how=tipo,
                               suffixes=("", "_cad"))
    print(f"   how='{tipo}':{'':<3}{len(resultado):>7,} linhas")

print("""
🔴 REPARE NO `inner`: ele PERDEU linhas silenciosamente.

   Os SKUs vendidos que não estão no cadastro simplesmente somem —
   e o faturamento total do relatório fica menor que o real.

   💭 É assim que "o Pandas dá um número e o SQL dá outro": um usou
      INNER JOIN e o outro LEFT JOIN, e ninguém percebeu.
""")

In [ ]:
# 🔑 `indicator=True` — a opção que deveria ser padrão
conferencia = esquerda.merge(produtos_incompleto, on="sku", how="left",
                             indicator=True, suffixes=("", "_cad"))
print("Conferência do merge:\n")
print(conferencia["_merge"].value_counts().to_string())

sem_cadastro = conferencia[conferencia["_merge"] == "left_only"]
print(f"\n🔴 {len(sem_cadastro):,} vendas de SKU que não está no cadastro")
print(f"   SKUs órfãos: {sorted(sem_cadastro['sku'].unique())[:5]} ...")

print("""
🧭 A REGRA: TODO MERGE DE PRODUÇÃO CONFERE O RESULTADO.

   assert len(depois) == len(antes), "o merge mudou o número de linhas"

   Ou, melhor ainda, `validate=`:
""")

try:
    esquerda.merge(produtos, on="sku", how="left", validate="many_to_one")
    print("   ✅ validate='many_to_one' passou — o cadastro não tem SKU repetido")
except Exception as erro:
    print(f"   🔴 {erro}")

In [ ]:
# 🔴 A multiplicação silenciosa
cadastro_com_duplicata = pd.concat([produtos.head(3), produtos.head(3)])
print(f"Um cadastro com SKU duplicado ({len(cadastro_com_duplicata)} linhas, "
      f"{cadastro_com_duplicata['sku'].nunique()} SKUs):\n")

antes = esquerda[esquerda["sku"].isin(produtos.head(3)["sku"])]
depois = antes.merge(cadastro_com_duplicata, on="sku", how="left",
                     suffixes=("", "_cad"))
print(f"   antes do merge : {len(antes):,} linhas")
print(f"   depois         : {len(depois):,} linhas   🔴 DOBROU")

soma_antes = (antes["quantidade"] * antes["preco_unitario"]).sum()
soma_depois = (depois["quantidade"] * depois["preco_unitario"]).sum()
print(f"\n   faturamento antes : R$ {soma_antes:>12,.2f}")
print(f"   faturamento depois: R$ {soma_depois:>12,.2f}   🔴 dobrou também")

print("""
🔴 ESTE É O BUG MAIS CARO DA AULA.

   Um SKU duplicado na tabela da DIREITA multiplica as linhas da
   esquerda. O faturamento dobra, e nenhum erro aparece.

   ✅ `validate="many_to_one"` pega isso na hora:
""")
try:
    antes.merge(cadastro_com_duplicata, on="sku", how="left",
                validate="many_to_one")
except Exception as erro:
    print(f"   🔴 {type(erro).__name__}: {erro}")

## 10. Séries temporais

In [ ]:
serie = pagas.set_index("data").sort_index()

diario = serie["receita"].resample("D").sum()
semanal = serie["receita"].resample("W").sum()
mensal = serie["receita"].resample("ME").sum()

print(f"`resample` reamostra pelo tempo:\n")
print(f"   diário  : {len(diario):>4} pontos")
print(f"   semanal : {len(semanal):>4} pontos")
print(f"   mensal  : {len(mensal):>4} pontos\n")
print("Últimos 5 meses:")
print(mensal.tail(5).to_string())

In [ ]:
# Média móvel: alisar o ruído do dia a dia
movel = diario.rolling(window=7, min_periods=7).mean()

comparativo = pd.DataFrame({
    "diario": diario,
    "media_7d": movel,
}).dropna().tail(8)
print("Receita diária vs média móvel de 7 dias:\n")
print(comparativo.to_string())

print("""
💡 `min_periods=7` é importante: sem ele, os primeiros dias teriam
   uma "média de 7 dias" calculada com 1, 2, 3 dias — um número que
   parece comparável e não é.

🔑 E a média móvel de 7 dias é a certa para varejo: ela elimina o
   efeito do dia da semana, que domina o ruído diário.
""")

In [ ]:
# 🔴 A armadilha do fuso horário
print("A coluna `data` tem fuso:\n")
print(f"   dtype: {vendas['data'].dtype}")
print(f"   exemplo: {vendas['data'].iloc[0]}")

em_sp = vendas["data"].dt.tz_convert("America/Sao_Paulo")
print(f"\n   em São Paulo: {em_sp.iloc[0]}")

dia_utc = vendas["data"].dt.date.value_counts().sort_index()
dia_sp = em_sp.dt.date.value_counts().sort_index()
diferentes = (dia_utc.reindex(dia_sp.index).fillna(0) != dia_sp).sum()

print(f"""
🔴 O MESMO INSTANTE, DOIS DIAS DIFERENTES.

   Uma venda às 22h de São Paulo é 01h do dia seguinte em UTC. Se o
   relatório agrupa por dia UTC e o financeiro conta por dia local,
   os totais diários NUNCA vão bater — e a diferença aparece
   justamente no fechamento do mês.

   Dias com contagem diferente entre UTC e São Paulo: {diferentes}

   🧭 A regra: GUARDE em UTC, APRESENTE no fuso local, e deixe
      explícito qual dos dois o relatório usa.
""")

## 🔧 Prática guiada — o relatório que bate com o SQL

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  O relatório de faturamento — feito para ser auditável
# ═══════════════════════════════════════════════════════════════
def faturamento_por_categoria(df: pd.DataFrame, fuso: str = "America/Sao_Paulo"):
    """Faturamento por categoria, com as decisões EXPLÍCITAS.

    🔑 Cada decisão abaixo tem um comentário dizendo o que ela faz com
       o número. É isso que permite comparar com o SQL e explicar a
       diferença — em vez de discutir qual planilha está certa.
    """
    dados = df.copy()

    # DECISÃO 1: só pedidos pagos (cancelado e pendente não são receita)
    dados = dados[dados["status"] == "pago"]

    # DECISÃO 2: receita = quantidade × preço. O FRETE NÃO ENTRA.
    dados["receita"] = dados["quantidade"] * dados["preco_unitario"]

    # DECISÃO 3: o dia é o do fuso LOCAL, não UTC
    dados["dia"] = dados["data"].dt.tz_convert(fuso).dt.date

    # DECISÃO 4: cidade nula vira grupo próprio (dropna=False)
    resultado = (dados.groupby("categoria", observed=True, dropna=False)
                 .agg(pedidos=("pedido_id", "nunique"),
                      itens=("quantidade", "sum"),
                      receita=("receita", "sum"))
                 .sort_values("receita", ascending=False))

    # 🔑 CONFERÊNCIA: a soma dos grupos tem que bater com o total
    total_direto = dados["receita"].sum()
    total_grupos = resultado["receita"].sum()
    assert abs(total_direto - total_grupos) < 0.01, (
        f"🔴 os grupos somam {total_grupos:,.2f}, o total é {total_direto:,.2f}")

    return resultado, {
        "linhas_entrada": len(df),
        "linhas_consideradas": len(dados),
        "linhas_descartadas": len(df) - len(dados),
        "receita_total": round(total_direto, 2),
        "fuso": fuso,
    }


relatorio, metadados = faturamento_por_categoria(vendas)
print(relatorio.to_string())
print("\nMetadados do cálculo:")
for chave, valor in metadados.items():
    print(f"   {chave:<22}{valor}")

> 🎯 **Os metadados são o que resolve a briga dos R$ 40 mil.**
>
> Quando o número do Pandas não bate com o do SQL, você não discute — você compara as decisões:
>
> | Pergunta | Pandas | SQL |
> |----------|--------|-----|
> | Inclui pendente? | não | ? |
> | Frete entra na receita? | não | ? |
> | O dia é UTC ou local? | local | ? |
> | Cidade nula é descartada? | não | ? |
>
> **Quase sempre a diferença está numa dessas quatro linhas** — e em cinco minutos você sabe qual.
>
> 💭 E o `assert` no meio da função não é paranoia: ele garante que a soma dos grupos bate com o total. Se um dia alguém adicionar um `groupby` que descarta nulos, o relatório falha em vez de mentir.

In [ ]:
# Salvando com os metadados junto
SAIDA = BASE / "faturamento.parquet"
relatorio.to_parquet(SAIDA)
(BASE / "faturamento.meta.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"✅ {SAIDA.name} + {SAIDA.stem}.meta.json")
print("\n💡 Todo número publicado deveria vir com a receita de como foi")
print("   calculado. Um Parquet e um JSON ao lado resolvem — e custam")
print("   duas linhas.")

## 📝 Exercícios

**E1.** Crie duas Series com índices parcialmente diferentes e some. Explique os `NaN`.

**E2.** Mostre a diferença entre `loc[10:30]` e `iloc[10:30]`. Explique por que uma inclui o fim.

**E3.** 🔴 Escreva um filtro com `and` em vez de `&` e explique o erro.

**E4.** 🔴 Reproduza o `SettingWithCopyWarning`. Corrija das duas formas e explique quando usar cada uma.

**E5.** Compare `mean()`, `sum()/count()` e `sum()/len()` numa coluna com nulos. Explique as diferenças.

**E6.** 🔴 Faça um `groupby` numa coluna com nulos e mostre que a soma dos grupos não bate com o total. Corrija.

**E7.** Escreva as decisões de tratamento de nulo para cada coluna do Atlas, com a justificativa de negócio.

**E8.** Defina a chave natural das vendas da Aurora e remova duplicatas por ela. Explique `keep='last'`.

**E9.** 🎯 Compare `iterrows`, `apply(axis=1)` e vetorizado em 100 mil linhas. Explique a diferença.

**E10.** Substitua um `apply` com `if/elif/else` por `np.select`.

**E11.** Use `agg` com nomes explícitos para calcular 5 métricas por categoria.

**E12.** 🔑 Use `transform` para calcular a participação de cada venda no total do seu grupo. Explique por que não serve `agg`.

**E13.** Faça o mesmo merge com `inner` e `left`. Explique a diferença no faturamento total.

**E14.** 🔴 Reproduza a multiplicação de linhas com uma chave duplicada à direita. Mostre `validate=` pegando.

**E15.** Use `indicator=True` para encontrar SKUs vendidos que não estão no cadastro.

**E16.** Reamostre a receita por dia, semana e mês. Calcule a média móvel de 7 dias.

**E17.** 🔴 Mostre que agrupar por dia em UTC e em horário local dá resultados diferentes.

**E18.** Escreva um relatório que devolva o resultado **e** os metadados das decisões.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

In [ ]:
# E17

In [ ]:
# E18

## 📋 Cola de referência

```python
# ═══ Seleção ═══
df.loc[rotulo, coluna]      por RÓTULO · fatia INCLUI o fim
df.iloc[posicao, n]         por POSIÇÃO · fatia exclui o fim
df[df["x"] > 10]            máscara booleana
# 🔴 use & | ~ (não and/or/not) e PARÊNTESES em toda condição

# ═══ 🔴 SettingWithCopy ═══
df.loc[mascara, "col"] = v      ✅ altera o ORIGINAL
recorte = df[mascara].copy()    ✅ novo objeto
df[mascara]["col"] = v          🔴 indexação encadeada — ambíguo

# ═══ 🔴 Nulos ═══
df.isna().sum()                 quantos por coluna
sum()/mean()/count()            IGNORAM nulo
len()                           CONTA
groupby(x)                      🔴 DESCARTA linhas com x nulo
groupby(x, dropna=False)        ✅ mantém
# 🎯 mean() = soma / COUNT, não soma / len

# ═══ Duplicatas ═══
df.duplicated(subset=["chave"]).sum()
df.drop_duplicates(subset=["chave"], keep="last")   # ordene antes!

# ═══ 🎯 Vetorização ═══
df["a"] * df["b"]                    ✅ vetorizado
np.where(cond, sim, nao)             2 opções
np.select([c1, c2], [v1, v2], default=v3)
df["x"].map(dicionario)
# 🔴 for/iterrows/apply(axis=1) = Python por linha, ordens de grandeza mais lento

# ═══ groupby ═══
.agg(nome=("coluna", "funcao"))      🔑 forma nomeada
.transform("sum")                     preserva o tamanho
.reset_index()                        tira as chaves do índice

# ═══ 🔴 merge ═══
how="inner"   perde o que não casa dos DOIS lados
how="left"    mantém tudo da esquerda
indicator=True                🔑 mostra o que casou
validate="many_to_one"        🔴 pega a MULTIPLICAÇÃO de linhas
assert len(depois) == len(antes)

# ═══ Tempo ═══
.resample("D"/"W"/"ME").sum()
.rolling(7, min_periods=7).mean()
.dt.tz_convert("America/Sao_Paulo")
# 🔴 guarde em UTC, apresente em local, e diga qual o relatório usa
```

## ✅ Checklist de saída

**Fundamentos**

- [ ] Entendo que a Series alinha por ÍNDICE, não por posição
- [ ] Distingo `loc` de `iloc`, inclusive nas fatias
- [ ] Uso `&`, `|`, `~` com parênteses

**Armadilhas**

- [ ] 🔴 **Sei corrigir o `SettingWithCopyWarning` das duas formas**
- [ ] Nunca uso indexação encadeada para atribuir
- [ ] 🔴 **Sei que `mean()` divide por `count()`, não por `len()`**
- [ ] 🔴 **Sei que `groupby` descarta chave nula por padrão**
- [ ] Confiro que a soma dos grupos bate com o total
- [ ] Trato nulo como decisão de negócio, documentada

**Desempenho**

- [ ] 🎯 **Vetorizo em vez de `apply(axis=1)`**
- [ ] Uso `np.where` e `np.select` para condicionais

**Combinação**

- [ ] Sei a diferença entre `inner` e `left` no total
- [ ] Uso `indicator=True` para conferir
- [ ] 🔴 **Uso `validate=` contra a multiplicação de linhas**

**Tempo**

- [ ] Uso `resample` e `rolling` com `min_periods`
- [ ] 🔴 **Sei que agrupar por dia UTC ≠ por dia local**

**Confiabilidade**

- [ ] Meus relatórios devolvem os metadados das decisões
- [ ] Sei explicar por que o meu número difere do SQL

---

### ➡️ Próxima aula

**`10_03_Alta_Performance_e_Polars.ipynb`** — Quando o Pandas começa a doer: memória, `dtypes`, Parquet, e o Polars com avaliação preguiçosa.